# 01 Data audit

Quick sniff test on the raw Arcep CSV. I want to know dtypes, null rates, 
weird values and the encoding situation before I touch anything.


In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))
%load_ext autoreload
%autoreload 2


In [2]:
import pandas as pd
from ftth_equity import config, io
df = io.load_raw()
print(df.shape)
df.head()


(57072, 20)


,x,y,imb_id,num_voie,cp_no_voie,type_voie,nom_voie,batiment,code_insee,code_poste,nom_com,catg_loc_imb,imb_etat,pm_ref,pm_etat,code_l331,geom_mod,type_imb,date_completude,date_completude_manquante
row_id,,,,,,,,,,,,,,,,,,,,
350698,4.087389,48.254920,IMB/10060/X/017J,20.0,<NA>,Rue,Olympe de Gouges,<NA>,10060,10450.0,BrÃ©viandes,individuel,signe,FI-10060-0006,deploye,FRTE,f,PA,NaN,NaN
350970,4.063110,48.291863,IMB/10387/X/0A1P,7.0,B,Rue,Rothier,<NA>,10387,10000.0,Troyes,entre 2 et 11,signe,FI-10387-000G,deploye,FRTE,f,IM,NaN,NaN
414894,4.027574,48.257070,IMB/10340/X/00Z9,100.0,<NA>,Rue,des Grives,<NA>,10340,10120.0,Saint-Germain,individuel,cible,FI-10340-0003,deploye,FRTE,f,PA,NaN,NaN
415103,4.027438,48.257022,IMB/10340/X/00ZA,102.0,<NA>,Rue,des Grives,<NA>,10340,10120.0,Saint-Germain,individuel,cible,FI-10340-0003,deploye,FRTE,f,PA,NaN,NaN
415976,4.026999,48.256873,IMB/10340/X/00ZB,104.0,<NA>,Rue,des Grives,<NA>,10340,10120.0,Saint-Germain,individuel,cible,FI-10340-0003,deploye,FRTE,f,PA,NaN,NaN


## Null rates


In [3]:
df.isna().mean().sort_values(ascending=False)


date_completude              1.000000
date_completude_manquante    1.000000
batiment                     0.952937
cp_no_voie                   0.899303
type_voie                    0.027404
imb_etat                     0.000350
y                            0.000000
x                            0.000000
num_voie                     0.000000
imb_id                       0.000000
code_insee                   0.000000
nom_voie                     0.000000
catg_loc_imb                 0.000000
nom_com                      0.000000
code_poste                   0.000000
pm_ref                       0.000000
code_l331                    0.000000
pm_etat                      0.000000
type_imb                     0.000000
geom_mod                     0.000000
dtype: float64

## Categorical cardinalities


In [4]:
for c in ['catg_loc_imb','imb_etat','pm_etat','code_l331','type_imb','geom_mod']:
    print(c, df[c].value_counts(dropna=False).head().to_dict())


catg_loc_imb {'individuel': 49103, 'entre 2 et 11': 6645, '12 ou plus': 1324}
imb_etat {'deploye': 53845, 'cible': 2458, 'signe': 558, 'en cours de deploiement': 71, 'abandonne': 65}
pm_etat {'deploye': 57064, 'en cours de deploiement': 8}
code_l331 {'FRTE': 36500, 'LOSA': 20572}
type_imb {'PA': 50362, 'IM': 6710}
geom_mod {'f': 57072}


## Coordinate sanity


In [5]:
df[['x','y']].describe()


,x,y
count,57072.000000,57072.000000
mean,4.068009,48.285075
std,0.075275,0.045905
min,3.793640,48.120868
25%,4.039460,48.263350
50%,4.068934,48.290010
75%,4.098979,48.307068
max,4.374820,48.468150


Conclusion: x/y are degrees (~4 E, 48 N), not Lambert-93. 
Encoding is cp1252 — see the mojibake on `nom_com`. Both handled by `io.load_raw`.